In [ ]:
"""
=============================================================
FILE 46 — STREAMING WORKFLOW
=============================================================

CONCEPTS TAUGHT
----------------
1. Streaming Workflows
2. Real-Time AI Responses
3. Progressive Rendering
4. Token Streaming
5. Interactive AI Systems
6. Live Workflow Updates
7. Streaming UX
8. Incremental Outputs
9. Responsive AI Interfaces
10. Real-Time Orchestration

CORE IDEA
-----------
Instead of waiting for the full response,
stream outputs progressively.

FLOW
-----
User Input
   ↓
Generate Tokens
   ↓
Stream Incrementally
   ↓
Real-Time UI

REAL WORLD USE CASES
---------------------
- ChatGPT-like interfaces
- coding copilots
- live AI dashboards
- real-time assistants
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

# ============================================================
# STEP 2 — LOAD ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — INITIALIZE MODEL
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    streaming=True
)

# ============================================================
# STEP 4 — DEFINE STATE
# ============================================================

class State(TypedDict):

    query: str

    response: str

# ============================================================
# STEP 5 — STREAMING NODE
# ============================================================

def streaming_node(state: State):

    print("\nStreaming Response:\n")

    final_response = ""

    for chunk in llm.stream(state["query"]):

        if chunk.content:
            print(chunk.content, end="", flush=True)
            final_response += chunk.content

    return {
        "response": final_response
    }

# ============================================================
# STEP 6 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node(
    "streaming_node",
    streaming_node
)

builder.add_edge(
    START,
    "streaming_node"
)

builder.add_edge(
    "streaming_node",
    END
)

# ============================================================
# STEP 7 — COMPILE GRAPH
# ============================================================

graph = builder.compile()

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 8 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "query":
        """
        Explain the future of Agentic AI
        in enterprise automation.
        """
    }
)

# ============================================================
# STEP 9 — FINAL RESPONSE
# ============================================================

print("\n\nFINAL STORED RESPONSE\n")
print("=" * 60)

print(result["response"])